### Hücre 1: Bağlantı ve tek gün

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from eptr2 import EPTR2

load_dotenv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\notepads\.env")
eptr = EPTR2(username=os.getenv("EPTR_USERNAME"),
             password=os.getenv("EPTR_PASSWORD"),
             use_dotenv=False)

gun = eptr.call("rt-cons", start_date="2024-01-15", end_date="2024-01-15")
print(type(gun))
print(gun.columns.tolist())
print(gun.head())
print("Satir sayisi:", len(gun))   # saatlikse 24 bekleniyor

<class 'pandas.core.frame.DataFrame'>
['date', 'time', 'consumption']
                        date   time  consumption
0  2024-01-15T00:00:00+03:00  00:00     34389.32
1  2024-01-15T01:00:00+03:00  01:00     32509.37
2  2024-01-15T02:00:00+03:00  02:00     30970.23
3  2024-01-15T03:00:00+03:00  03:00     30183.43
4  2024-01-15T04:00:00+03:00  04:00     30066.57
Satir sayisi: 24


### Hücre 2: Verinin ne kadar geriye gittiği

In [3]:
for d in ["2010-01-15", "2012-01-15", "2014-01-15", "2015-01-15", "2016-01-15", "2017-01-15"]:
    try:
        r = eptr.call("rt-cons", start_date=d, end_date=d)
        print(d, "->", len(r), "satir")
    except Exception as e:
        print(d, "-> HATA:", str(e)[:150])

2010-01-15 -> 0 satir
2012-01-15 -> 0 satir
2014-01-15 -> 0 satir
2015-01-15 -> 0 satir
2016-01-15 -> 0 satir
2017-01-15 -> 24 satir


### Hücre 3: Bir yıllık indirme (aylık parçalar halinde)

In [4]:
parcalar = []
for ay in pd.date_range("2024-01-01", "2024-12-01", freq="MS"):
    son = (ay + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    parcalar.append(eptr.call("rt-cons", start_date=ay.strftime("%Y-%m-%d"), end_date=son))
yil = pd.concat(parcalar, ignore_index=True)
print("2024 satir sayisi:", len(yil))   # 2024 artik yil: saatlikse 8.784 bekleniyor
yil.to_csv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\data\tr_tuketim_2024_test.csv", index=False)

2024 satir sayisi: 8784


### Adım 6 — Sıcaklık Verisi Testi (VS Code, aynı notebook)

In [5]:
import requests
r = requests.get(
    "https://www.ncei.noaa.gov/cdo-web/api/v2/stations",
    headers={"token": os.getenv("NOAA_TOKEN")},
    params={"locationid": "FIPS:TU", "datasetid": "GHCND", "limit": 1000},
    timeout=60,
)
print(r.status_code)
st = pd.DataFrame(r.json().get("results", []))
print(len(st), "istasyon")
print(st[st["name"].str.contains("ANKARA|ISTANBUL|IZMIR", case=False, na=False)][["id", "name", "mindate", "maxdate", "datacoverage"]])

200
58 istasyon
                   id                 name     mindate     maxdate  \
1   GHCND:TU000017062         ISTANBUL, TU  1929-01-01  2007-05-09   
27  GHCND:TUM00017130   ANKARA CENTRAL, TU  2002-12-31  2026-09-22   
40  GHCND:TUM00017220  IZMIR GUZELYALI, TU  1950-01-15  2026-09-22   
56  GHCND:TUW00033207            IZMIR, TU  1963-03-01  1967-12-31   

    datacoverage  
1         0.9589  
27        0.8169  
40        0.3702  
56        1.0000  


### Hücre 9: Ankara günlük sıcaklık verisini indirme (2016-2026)

In [6]:
import time
import requests

def noaa_gunluk(istasyon, baslangic, bitis, token):
    satirlar, offset = [], 1
    while True:
        r = requests.get(
            "https://www.ncei.noaa.gov/cdo-web/api/v2/data",
            headers={"token": token},
            params={"datasetid": "GHCND", "stationid": istasyon,
                    "datatypeid": ["TMAX", "TMIN", "TAVG"],
                    "startdate": baslangic, "enddate": bitis,
                    "units": "metric", "limit": 1000, "offset": offset},
            timeout=60,
        )
        r.raise_for_status()
        j = r.json()
        satirlar += j.get("results", [])
        toplam = j.get("metadata", {}).get("resultset", {}).get("count", 0)
        if offset + 1000 > toplam:
            break
        offset += 1000
        time.sleep(0.3)
    return pd.DataFrame(satirlar)

ISTASYON = "GHCND:TUM00017130"
parcalar = []
for yil_no in range(2016, 2027):
    bas = f"{yil_no}-01-01"
    bit = "2026-09-22" if yil_no == 2026 else f"{yil_no}-12-31"
    df_y = noaa_gunluk(ISTASYON, bas, bit, os.getenv("NOAA_TOKEN"))
    print(yil_no, "->", len(df_y), "kayit")
    parcalar.append(df_y)
    time.sleep(0.3)

ankara = pd.concat(parcalar, ignore_index=True)
ankara.to_csv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\data\ankara_noaa_ham.csv", index=False)

2016 -> 965 kayit
2017 -> 992 kayit
2018 -> 795 kayit
2019 -> 638 kayit
2020 -> 682 kayit
2021 -> 990 kayit
2022 -> 994 kayit
2023 -> 1004 kayit
2024 -> 971 kayit
2025 -> 1077 kayit
2026 -> 774 kayit


### Hücre 10: Yıllık kapsam kontrolü

In [7]:
ankara["date"] = pd.to_datetime(ankara["date"])
tablo = ankara.pivot_table(index="date", columns="datatype", values="value")
tum_gunler = pd.date_range("2016-01-01", "2026-09-22", freq="D")
tablo = tablo.reindex(tum_gunler)
print("Toplam gun:", len(tum_gunler))
print("Eksik gun sayisi:")
print(tablo.isna().sum())
print("\nYillara gore eksik gun (TMAX / TMIN):")
print(tablo[["TMAX", "TMIN"]].isna().groupby(tablo.index.year).sum())

Toplam gun: 3918
Eksik gun sayisi:
datatype
TAVG      30
TMAX    1042
TMIN     800
dtype: int64

Yillara gore eksik gun (TMAX / TMIN):
datatype  TMAX  TMIN
2016        47    84
2017        14    89
2018       216    83
2019       362    95
2020       326    90
2021        18    84
2022        10    91
2023        15    76
2024        21    95
2025         6     6
2026         7     7


### Hücre 11: TAVG kontrolü ve eksiklerin doldurulması

In [8]:
t = tablo["TAVG"].copy()
print("TAVG eksik:", t.isna().sum(), "/", len(t))

# 1) Iklim kontrolu: aylik ortalamalar makul mu (kis soguk, yaz sicak)?
print(t.groupby(t.index.month).mean().round(1))

# 2) TMAX ve TMIN'in ikisinin de oldugu gunlerde TAVG ile (TMAX+TMIN)/2 farki
ortak = tablo.dropna(subset=["TMAX", "TMIN", "TAVG"])
fark = ortak["TAVG"] - (ortak["TMAX"] + ortak["TMIN"]) / 2
print("Ortak gun:", len(ortak), "| ortalama fark:", round(fark.mean(), 2), "| std:", round(fark.std(), 2))

# 3) En uzun ardisik eksik blok
bosluk = t.isna().astype(int).groupby(t.notna().cumsum()).sum()
print("En uzun ardisik eksik:", bosluk.max(), "gun")

# 4) Kisa bosluklari (en fazla 3 gun) dogrusal doldur
t_dolu = t.interpolate(limit=3)
print("Doldurma sonrasi kalan eksik:", t_dolu.isna().sum())
t_dolu.to_frame("tavg_c").to_csv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\data\ankara_tavg_gunluk.csv")

TAVG eksik: 30 / 3918
1      2.2
2      4.7
3      7.6
4     12.6
5     16.6
6     21.4
7     24.9
8     25.6
9     20.7
10    14.6
11     8.9
12     4.5
Name: TAVG, dtype: float64
Ortak gun: 2369 | ortalama fark: 0.05 | std: 0.73
En uzun ardisik eksik: 7 gun
Doldurma sonrasi kalan eksik: 7


(bitirme-proje) C:\Deep_Learning_to_AI_Agents>eptr2 describe eak
eak: Available Installed Capacity (AIC)
  Category: Üretim Planlama
  Description: Available Installed Capacity: The active power capacity that a generation unit can provide to the system.
  Method: POST
  Path: electricity-service/v1/generation/data/aic
  Required params: ['start_date', 'end_date', 'region']
  Optional params: ['org_id', 'uevcb_id']
  URL: https://seffaflik.epias.com.tr/electricity/electricity-generation/planning/available-installed-capacity-aic

Example: eptr2 call eak --start-date <value> --end-date <value>

### Adım A: Sıcaklık boşluklarını tamamlama

In [9]:
# Kalan 7 eksik gun nerede?
kalan = t_dolu[t_dolu.isna()]
print("Kalan eksik gunler:", list(kalan.index.date))

# Kalanlari "o gunun cok yillik ortalamasi" (klimatoloji) ile doldur, bayrakla isaretle
gun_no = t_dolu.index.dayofyear
klima = t_dolu.groupby(gun_no).transform("mean")
tavg_son = t_dolu.fillna(klima)
bayrak = t.isna().astype(int)          # 1 = orijinal veride eksikti (doldurulmus)

print("Son eksik sayisi:", tavg_son.isna().sum())
print("Doldurulan toplam gun:", bayrak.sum())
pd.DataFrame({"tavg_c": tavg_son, "tavg_doldurma": bayrak}).to_csv(
    r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\data\ankara_tavg_gunluk.csv")

Kalan eksik gunler: [datetime.date(2024, 3, 31), datetime.date(2024, 11, 21), datetime.date(2025, 9, 2), datetime.date(2026, 3, 29), datetime.date(2026, 3, 30), datetime.date(2026, 3, 31), datetime.date(2026, 4, 1)]
Son eksik sayisi: 0
Doldurulan toplam gun: 30


(bitirme-proje) C:\Users\ozaim>eptr2 search bölge

## Listeleme
----------------------------------------
  mms-region-list              Market Message System Region List
  region-list                  Region Listing

## Talep Tahmini
----------------------------------------
  long-term-demand-forecast    Demand Forecast

Total: 3 endpoints in 2 categories

(bitirme-proje) C:\Users\ozaim>eptr2 search region

## Listeleme
----------------------------------------
  mms-region-list              Market Message System Region List
  region-list                  Region Listing

## Talep Tahmini
----------------------------------------
  long-term-demand-forecast    Demand Forecast

## Uncategorized
----------------------------------------
  distribution-region-list

Total: 4 endpoints in 3 categories


### Bölge listesini döndüren servisin anahtarı
(bitirme-proje) C:\Users\ozaim>eptr2 describe region-list
region-list: Region Listing
  Category: Listeleme
  Description: Region Listing
  Method: GET
  Path: electricity-service/v1/generation/data/region-list
  Required params: []
  Optional params: []
  URL: https://seffaflik.epias.com.tr/electricity-service/technical/tr/index.html#_region-list

Example: eptr2 call region-list

In [10]:
bolgeler = eptr.call("region-list")
print(bolgeler)

   regionId regionShortName
0         1             TR1


### Hücre 14: EAK testi

In [11]:
REGION = "TR1"   # Hucre 13 ciktisiyla dogrulayin

eak = eptr.call("eak", start_date="2026-09-26", end_date="2026-09-26", region=REGION)
print(eak.columns.tolist())
print(eak.head(3))
print("Satir:", len(eak))

['date', 'time', 'toplam', 'dogalgaz', 'ruzgar', 'linyit', 'tasKomur', 'ithalKomur', 'fuelOil', 'jeotermal', 'barajli', 'nafta', 'biokutle', 'akarsu', 'gunes', 'diger']
                        date   time    toplam  dogalgaz   ruzgar   linyit  \
0  2026-09-26T00:00:00+03:00  00:00  57568.66  15231.96  6376.15  4209.29   
1  2026-09-26T01:00:00+03:00  01:00  57125.97  15189.34  6264.07  4211.29   
2  2026-09-26T02:00:00+03:00  02:00  57008.41  15197.74  6150.03  4212.29   

   tasKomur  ithalKomur  fuelOil  jeotermal   barajli  nafta  biokutle  \
0       119      8819.7   232.76    1134.39  15794.34      0   1257.56   
1       119      8819.7   232.76    1142.49  15518.59      0   1259.86   
2       119      8819.7   232.76    1152.58  15567.59      0   1258.86   

    akarsu   gunes   diger  
0  3622.86  474.52  296.13  
1  3599.88  472.86  296.13  
2  3523.60  471.63  302.63  
Satir: 24


### Hücre 18: Gerçek kapasiteye göre marj (ilk test)

In [12]:
tuk = eptr.call("rt-cons", start_date="2026-09-26", end_date="2026-09-26")

m = tuk[["date", "consumption"]].merge(eak[["date", "toplam", "gunes", "ruzgar"]], on="date", how="inner")
m["marj_kapasite"] = (m["toplam"] - m["consumption"]) / m["toplam"]     # PJM projesindeki kod tanimi
m["marj_yuk"] = (m["toplam"] - m["consumption"]) / m["consumption"]     # IRM tanimi
print("Eslesen saat:", len(m), "(24 bekleniyor)")
print(m[["date", "consumption", "toplam", "gunes", "marj_kapasite", "marj_yuk"]].round(3).to_string())
en_dar = m.loc[m["marj_kapasite"].idxmin()]
print("\nEn dar saat:", en_dar["date"], "| tuketim:", en_dar["consumption"], "| EAK:", en_dar["toplam"],
      "| marj (kapasite bazli): %", round(en_dar["marj_kapasite"] * 100, 1))

Eslesen saat: 24 (24 bekleniyor)
                         date  consumption    toplam     gunes  marj_kapasite  marj_yuk
0   2026-09-26T00:00:00+03:00     36802.47  57568.66    474.52          0.361     0.564
1   2026-09-26T01:00:00+03:00     35121.61  57125.97    472.86          0.385     0.627
2   2026-09-26T02:00:00+03:00     33675.94  57008.41    471.63          0.409     0.693
3   2026-09-26T03:00:00+03:00     32195.19  58074.53    472.09          0.446     0.804
4   2026-09-26T04:00:00+03:00     32512.30  57335.89    479.37          0.433     0.764
5   2026-09-26T05:00:00+03:00     32905.30  57311.00    506.60          0.426     0.742
6   2026-09-26T06:00:00+03:00     33885.95  59007.78   1070.42          0.426     0.741
7   2026-09-26T07:00:00+03:00     35419.39  61287.00   3671.83          0.422     0.730
8   2026-09-26T08:00:00+03:00     39175.69  65539.05   8196.29          0.402     0.673
9   2026-09-26T09:00:00+03:00     41160.91  69378.62  11950.27          0.407     0.686

### Adım C: EAK geçmişi

In [13]:
for d in ["2016-06-15", "2017-01-15", "2019-01-15", "2022-01-15"]:
    try:
        r = eptr.call("eak", start_date=d, end_date=d, region=REGION)
        print(d, "->", len(r), "satir")
    except Exception as e:
        print(d, "-> HATA:", str(e)[:150])

2016-06-15 -> 24 satir
2017-01-15 -> 24 satir
2019-01-15 -> 24 satir
2022-01-15 -> 24 satir


### Adım D: Tüketimin tam başlangıç ayı

In [14]:
for ay in pd.date_range("2016-01-01", "2016-12-01", freq="MS"):
    g = ay.strftime("%Y-%m-%d")
    r = eptr.call("rt-cons", start_date=g, end_date=g)
    print(g, "->", len(r), "satir")

2016-01-01 -> 0 satir
2016-02-01 -> 0 satir
2016-03-01 -> 0 satir
2016-04-01 -> 24 satir
2016-05-01 -> 24 satir
2016-06-01 -> 24 satir
2016-07-01 -> 24 satir
2016-08-01 -> 24 satir
2016-09-01 -> 24 satir
2016-10-01 -> 24 satir
2016-11-01 -> 24 satir
2016-12-01 -> 24 satir


### Adım E: 2024 tüketim verisinin kalite kontrolü

In [15]:
yil = pd.read_csv(r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\data\tr_tuketim_2024_test.csv")
yil["date"] = pd.to_datetime(yil["date"])
print("Aralik:", yil["date"].min(), "->", yil["date"].max())
print("Tekrar eden saat:", yil["date"].duplicated().sum())
tam = pd.date_range(yil["date"].min(), yil["date"].max(), freq="h")
print("Eksik saat:", len(tam) - yil["date"].nunique())
print("Bos deger:", yil["consumption"].isna().sum())
print(yil["consumption"].describe().round(0))
fark = yil["consumption"].diff().abs()
print("En buyuk 5 saatlik degisim:")
print(yil.loc[fark.nlargest(5).index, ["date", "consumption"]])

Aralik: 2024-01-01 00:00:00+03:00 -> 2024-12-31 23:00:00+03:00
Tekrar eden saat: 0
Eksik saat: 0
Bos deger: 0
count     8784.0
mean     39013.0
std       6494.0
min      20062.0
25%      33773.0
50%      39170.0
75%      43123.0
max      57772.0
Name: consumption, dtype: float64
En buyuk 5 saatlik degisim:
                          date  consumption
5720 2024-08-26 08:00:00+03:00     43370.69
5384 2024-08-12 08:00:00+03:00     45089.77
5552 2024-08-19 08:00:00+03:00     44439.61
5216 2024-08-05 08:00:00+03:00     45126.39
4784 2024-07-18 08:00:00+03:00     47382.12


### Aksiyon Planı: Faz 1 (Veri Hazırlığı)

In [16]:
for d in ["2016-03-15", "2016-04-01", "2016-05-01"]:
    try:
        r = eptr.call("eak", start_date=d, end_date=d, region=REGION)
        print(d, "->", len(r), "satir")
    except Exception as e:
        print(d, "-> HATA:", str(e)[:150])

2016-03-15 -> 0 satir
2016-04-01 -> 24 satir
2016-05-01 -> 24 satir


### Hücre 20: 2024'ün uç değerleri gerçek mi?

In [18]:
for etiket, idx in [("En dusuk", yil["consumption"].idxmin()), ("En yuksek", yil["consumption"].idxmax())]:
    print(f"\n{etiket}: {yil.loc[idx, 'date']} -> {yil.loc[idx, 'consumption']:.0f} MWh")
    print(yil.iloc[max(idx - 3, 0):idx + 4][["date", "consumption"]].to_string(index=False))

fark = yil["consumption"].diff()
i = fark.abs().idxmax()
print("\nEn buyuk degisim ve komsu saatleri:")
print(yil.iloc[i - 3:i + 4][["date", "consumption"]].to_string(index=False))


En dusuk: 2024-04-11 07:00:00+03:00 -> 20062 MWh
                     date  consumption
2024-04-11 04:00:00+03:00     20791.83
2024-04-11 05:00:00+03:00     20765.66
2024-04-11 06:00:00+03:00     20076.64
2024-04-11 07:00:00+03:00     20062.44
2024-04-11 08:00:00+03:00     21209.24
2024-04-11 09:00:00+03:00     22886.51
2024-04-11 10:00:00+03:00     24197.28

En yuksek: 2024-07-23 14:00:00+03:00 -> 57772 MWh
                     date  consumption
2024-07-23 11:00:00+03:00     56135.29
2024-07-23 12:00:00+03:00     55210.31
2024-07-23 13:00:00+03:00     56612.47
2024-07-23 14:00:00+03:00     57772.40
2024-07-23 15:00:00+03:00     57516.72
2024-07-23 16:00:00+03:00     56907.86
2024-07-23 17:00:00+03:00     55303.16

En buyuk degisim ve komsu saatleri:
                     date  consumption
2024-08-26 05:00:00+03:00     33845.77
2024-08-26 06:00:00+03:00     32519.08
2024-08-26 07:00:00+03:00     35549.01
2024-08-26 08:00:00+03:00     43370.69
2024-08-26 09:00:00+03:00     48156.71
2024